Instalando bibliotecas e dependências

In [3]:
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import numpy as np
import qiskit
import math
from math import sqrt
import matplotlib.pyplot as plt
from qiskit.visualization import array_to_latex

Matplotlib is building the font cache; this may take a moment.


# Caminho teste do algoritmo DaC para um vetor de 2 qubits (4 estados)

Aqui está o passo a passo de como a função gen_angles percorre um vetor hipotético de tamanho N = 4, retornando um vetor de ângulos normalizados

In [ ]:
target = [sqrt(6), 0, 0, sqrt(4)]

#gen_angles na 1a chamada (gen_angles1)
new_x1 = list(range(2))  #vetor auxiliar de tamanho N/2 (4/2 = 2 qubits)

for k in range(len(new_x)):
    new_x1[k] = sqrt(target[2*k]**2 + target[2*k]**2)


#gen_angles na 2a chamada (gen_angles2)
new_x2 = list(1)  #N/2 = 2/2 = 1 

for k in range(len(new_x2)):
    new_x2[k] = sqrt(new_x1[2*k]**2 + new_x1[2*k]**2)

#gen_angles na 3a chamada (gen_angles3)
new_x3 = list(range(1/2)) #valor menor que 1, retorna na recursãao

#volta para gen_angles2

inner_anglers = [] #aqui inner_angles = 0
angles2 = list(range[1]) #N/2 = 2/2 = 1
for k in range(len(new_x2)):
    if new_x2[k] != 0:
        if new_x1[2*k] > 0:
            angles2[k] = 2 * np.arcsin(new_x1[2*k+1] / new_x2[k])
        else:
            angles2[k] = 2 * np.pi - 2 * np.arcsin(new_x1[2*k+1] / new_x2[k])

    else:
        angles2[k] = 0

angles2 = 0 + angles2


#volta para gen_angles1

inner_angles = angles2
angles1 = list(range(2)) #N/2 = 4/2 = 2
for k in range(len(new_x1)):
    if new_x1[k] != 0:
        if target[k] > 0:
            angles1[k] = 2 * np.arcsin(target[2*k+1] / new_x1[k])
        else:
            angles1[k] = 2 * np.pi - 2 * np.arcsin(target[2*k+1] / new_x1[k])
    else:
        angles1[k] = 0


angles1 = inner_angles + angles1 #output







    



# Implementação do gen_circuit

A função `gen_circuit` constrói o circuito quântico de preparação de estados com base nos ângulos calculados pela função `gen_angles`.

In [ ]:
def gen_circuit(angles):
    """
    Gera um circuito quântico de preparação de estados com base nos ângulos calculados.
    
    Parâmetros:
    angles (list): Vetor de dimensão N - 1 contendo os ângulos gerados por gen_angles.
    
    Retorna:
    QuantumCircuit: Circuito quântico gerado.
    """
    import math
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import RYGate

    N = len(angles) + 1
    n = int(math.log2(N))
    circuit = QuantumCircuit(n)
    q = circuit.qubits

    def level(k):
        """Retorna o nível (altura) do índice k na árvore binária de ângulos (0-indexed)."""
        return int(math.log2(k + 1))

    def index(k, j, q, circuit):
        """
        Aplica portas X nos qubits de controle para representar a condição de controle-0
        (círculo branco) em posições adequadas.
        """
        s = k - (2**j - 1)
        for i in range(j):
            # Obtém o bit correspondente a q[i] na representação de s de j bits.
            # O bit de q[i] corresponde ao bit (j - 1 - i) de s (MSB em q[0], LSB em q[j-1]).
            bit = (s >> (j - 1 - i)) & 1
            if bit == 0:
                circuit.x(q[i])

    for k in range(N - 1):
        j = level(k)
        
        # Mapeia os controles com círculos brancos para controle 1 aplicando X
        index(k, j, q, circuit)
        
        # Aplica a rotação controlada (CRy)
        angle = angles[k]
        if j == 0:
            circuit.ry(angle, q[0])
        else:
            controlled_ry = RYGate(angle).control(num_ctrl_qubits=j)
            circuit.append(controlled_ry, [*q[:j], q[j]])
            
        # Desfaz as portas X para restaurar os estados originais dos qubits
        index(k, j, q, circuit)

    return circuit

# Teste e Validação da Função gen_circuit

Vamos instanciar a função `gen_circuit` com os ângulos do exemplo de 8 dimensões (3 qubits) descritos na Seção 2 do artigo para conferir visualmente a estrutura gerada.

In [ ]:
# Ângulos obtidos a partir do exemplo do artigo para N=8 (3 qubits)
test_angles = [1.98, 1.91, 1.43, 1.98, 1.05, 2.09, 1.23]

# Gerar o circuito quântico
qc = gen_circuit(test_angles)

# Desenhar e exibir o circuito
print("Estrutura do Circuito Quântico Gerado (N=8, 3 qubits):")
print(qc.draw(output='text'))